In [3]:
from pathlib import Path
import pandas as pd

results_path = Path().resolve().parent / "experiments"
models = [
    "L1-Qwen3-8B-Max",
    "Qwen3-8B",
    "L1-Qwen-1.5B-Exact",
    "TokenSkip-Qwen2",
    "QwQ-32B-thinkprune-iter2k",
    "LCR1_7B",
]

datasets = ["math-500", "gsm8k", "olympiad", "amc", "aime-250"]

rows = []
for model in models:
    model_dir = results_path / model
    for dataset in datasets:
        parquet_file = model_dir / f"{dataset}_results.parquet"
        if not parquet_file.exists():
            continue
        df = pd.read_parquet(parquet_file)
        total = len(df)
        correct = df["is_correct"].sum()
        accuracy = correct / total if total > 0 else 0.0
        avg_tokens = df["token_count"].mean()
        rows.append({
            "model": model,
            "dataset": dataset,
            "accuracy": accuracy,
            "num_correct": correct,
            "num_total": total,
            "avg_tokens": avg_tokens,
        })

summary = pd.DataFrame(rows)
summary

,model,dataset,accuracy,num_correct,num_total,avg_tokens
0,L1-Qwen3-8B-Max,math-500,0.731463,365,499,2429.765531
1,L1-Qwen3-8B-Max,gsm8k,0.805914,1063,1319,2137.841547
2,L1-Qwen3-8B-Max,olympiad,0.344214,232,674,3119.321958
3,L1-Qwen3-8B-Max,amc,0.725000,29,40,2749.200000
4,L1-Qwen3-8B-Max,aime-250,0.380000,95,250,3406.496000
5,Qwen3-8B,math-500,0.757515,378,499,5895.448898
6,Qwen3-8B,gsm8k,0.900682,1188,1319,5379.167551
7,Qwen3-8B,olympiad,0.292285,197,674,5954.000000
8,Qwen3-8B,amc,0.625000,25,40,5920.675000
9,Qwen3-8B,aime-250,0.228000,57,250,6000.840000


In [4]:
for model in models:
    model_data = summary[summary["model"] == model]
    if model_data.empty:
        print(f"\n{model}: No results found\n")
        continue
    print(f"\n{'='*70}")
    print(f"Model: {model}")
    print(f"{'='*70}")
    print(f"{'Dataset':<15} {'Accuracy':>12} {'Correct':>10} {'Total':>8} {'Avg Tokens':>12}")
    print(f"{'-'*70}")
    for _, row in model_data.iterrows():
        print(f"{row['dataset']:<15} {row['accuracy']*100:>11.2f}% {row['num_correct']:>10} {row['num_total']:>8} {row['avg_tokens']:>12.1f}")
    total_correct = model_data["num_correct"].sum()
    total_problems = model_data["num_total"].sum()
    total_tokens = (model_data["avg_tokens"] * model_data["num_total"]).sum()
    overall_acc = total_correct / total_problems if total_problems > 0 else 0
    overall_avg = total_tokens / total_problems if total_problems > 0 else 0
    print(f"{'-'*70}")
    print(f"{'OVERALL':<15} {overall_acc*100:>11.2f}% {total_correct:>10} {total_problems:>8} {overall_avg:>12.1f}")
    print(f"{'='*70}")


Model: L1-Qwen3-8B-Max
Dataset             Accuracy    Correct    Total   Avg Tokens
----------------------------------------------------------------------
math-500              73.15%        365      499       2429.8
gsm8k                 80.59%       1063     1319       2137.8
olympiad              34.42%        232      674       3119.3
amc                   72.50%         29       40       2749.2
aime-250              38.00%         95      250       3406.5
----------------------------------------------------------------------
OVERALL               64.13%       1784     2782       2550.8

Model: Qwen3-8B
Dataset             Accuracy    Correct    Total   Avg Tokens
----------------------------------------------------------------------
math-500              75.75%        378      499       5895.4
gsm8k                 90.07%       1188     1319       5379.2
olympiad              29.23%        197      674       5954.0
amc                   62.50%         25       40       5920.7
ai

In [5]:

# Load all AIME and Olympiad wrong answers across all models
target_datasets = ["olympiad"]

wrong_answers = []
for model in models:
    model_dir = results_path / model
    for dataset in target_datasets:
        parquet_file = model_dir / f"{dataset}_results.parquet"
        if not parquet_file.exists():
            continue
        df = pd.read_parquet(parquet_file)
        wrong = df[df["is_correct"] == False].copy()
        wrong["model"] = model
        wrong["dataset"] = dataset
        wrong_answers.append(wrong)

wrong_df = pd.concat(wrong_answers, ignore_index=True)

# Display summary of wrong answers per model/dataset
print("Wrong answers per model/dataset:")
print(wrong_df.groupby(["model", "dataset"]).size().unstack(fill_value=0).to_string())
print(f"\nTotal wrong answers: {len(wrong_df)}")



Wrong answers per model/dataset:
dataset                    olympiad
model                              
L1-Qwen3-8B-Max                 442
LCR1_7B                         414
QwQ-32B-thinkprune-iter2k       378
Qwen3-8B                        477
TokenSkip-Qwen2                 488

Total wrong answers: 2199


In [16]:
from pathlib import Path
import pandas as pd
from eval_pipeline import is_equiv
df_olympiad_lcr1 = pd.read_parquet(Path().resolve().parent / "experiments" / "LCR1_7B"/ "olympiad_results.parquet")


In [32]:
import re
def extract_boxed(s) -> str | None:
    print(s)
    if not s:
        return None
    # MATH & AIME
    matches = re.findall(r"(?<=boxed)\{([^}]*)\}", s)
    if matches:
        return ", ".join(m.strip() for m in matches)
    # GSM8K
    matches = re.findall(r"(?m)^[ \t]*####[ \t]*([^\n\r#]+?)[ \t]*$", s)
    if matches:
        return ", ".join(m.strip() for m in matches)
    # Olympiad
    matches = re.findall(r"\$([^$]*)\$", s)
    if matches:
        return ", ".join(m.strip() for m in matches)
    # AMC
    matches = re.findall(r"(?m)^[ \t]*([+-]?\d+(?:\.\d+)?)[ \t]*$", s)
    if matches:
        return ", ".join(m.strip() for m in matches)
    return s

In [13]:
null_samples = df_olympiad_lcr1[df_olympiad_lcr1["expected_value"].isnull()]

In [33]:
null_samples['expected_value'] = null_samples['solution'].apply(extract_boxed)
null_samples['generated_value'] = null_samples['generated_value'].apply(extract_boxed)
null_samples['is_correct'] = is_equiv(null_samples["expected_value"], null_samples["generated_value"])

1,3,5
1,3
2,3,4
2500,7500
1,2,3,4
1,2
11,13
10,11,12,13,14,-2,-1,0,1,2
-1,0,1
(-\infty,-5)
62.5%
(6,5)
5,1
7,14,18
1,-1,7,-7
(-7,-1)
1,2,5,8,9
-3,0
(4,2,3,2,4,3,1,1),(1,1,3,4,2,3,2,4),(4,1,1,3,4,2,3,2),(2,3,2,4,3,1,1,4),(3,4,2,3,2,4,1,1),(1,1,4,2,3,2,4,3)
(7,5,1,1,9,3,5,7,3,8,6,4,2,9,2,4,6,8)
1, 3, 5, 15
-2,-1,1,2
(1,-4,-2),(3,2,3),(13,2,-2)
(1,7,103, 105), (3, 5, 101, 107)
17.5%
16,3,1,1
7,4
-6,-8,-10
2,4,3
5,15
-5,7
1,3,9,11
11,19,37,73
8,32,44,68,80
19,0,1
2400, 1350, 864
-2,-2
3,4
(2,32), (8,18)
27,54,108,135,189,216,270,297,351,378
1,2
1,3
nan


TypeError: expected string or bytes-like object, got 'float'

In [25]:
null_samples.head(10)

,unique_id,problem,solution,generated,expected_value,generated_value,token_count,is_correct
21,14763,"For each positive integer $k$, let $t(k)$ be t...","1,3,5","<think>\nOkay, so I have this problem where fo...","1,3,5",NaN,6000,False
37,14779,Find all positive integers $n$ with the follow...,"1,3","<think>\nOkay, so I have this problem where I ...","1,3",NaN,6000,False
81,14823,Find all positive integers $n \geqslant 2$ for...,"2,3,4","<think>\nOkay, so I have this problem where I ...","2,3,4",NaN,6000,False
86,14828,Let $\mathbb{Z}_{\geqslant 0}$ be the set of n...,"2500,7500","<think>\nOkay, so I have this problem where I ...","2500,7500",NaN,6000,False
104,14846,Find all positive integers $n$ such that there...,"1,2,3,4","<think>\nOkay, so I have this problem where I ...","1,2,3,4",NaN,6000,False
117,14859,"Let $m>1$ be an integer. A sequence $a_{1}, a_...","1,2","<think>\nOkay, so I have this problem where I ...","1,2",2,4454,False
122,14864,Determine all integers $m$ for which the $m \t...,"11,13","<think>\nOkay, so I have this problem where I ...","11,13",NaN,6000,False
125,14867,There are two increasing sequences of five con...,"10,11,12,13,14,-2,-1,0,1,2","<think>\nOkay, so I have this problem where I ...","10,11,12,13,14,-2,-1,0,1,2","[-2, -1, 0, 1, 2]",1223,False
127,14869,Determine all integer values of $x$ such that ...,"-1,0,1","<think>\nOkay, so I need to figure out all the...","-1,0,1",-1,693,False
131,14873,"Determine, with justification, all values of $...","(-\infty,-5)","<think>\nOkay, so I need to figure out for whi...","(-\infty,-5)",k < -5,887,False


In [19]:
print(null_samples["generated"].iloc[8])
null_samples["is_correct"].sum() / null_samples.shape[0]

<think>
Okay, so I need to figure out all the integer values of x that satisfy the inequality (x² - 3)(x² + 5) < 0. Hmm, let me break this down step by step.

First, I remember that when dealing with inequalities involving products, the product will be negative if one factor is positive and the other is negative. So, I need to find the values of x where one of the factors is positive and the other is negative.

Let me write down the two factors separately:

1. Factor 1: x² - 3
2. Factor 2: x² + 5

I should analyze each factor to see when they are positive or negative.

Starting with Factor 1: x² - 3.

This is a quadratic expression. To find when it's positive or negative, I can solve the equation x² - 3 = 0.

x² - 3 = 0
x² = 3
x = ±√3

So, the critical points are at x = √3 and x = -√3. Since √3 is approximately 1.732, these points divide the number line into intervals where the expression x² - 3 is either positive or negative.

Testing intervals:

- For x < -√3, say x = -2: (-2)² - 3 =

np.float64(0.023809523809523808)